# V6.2E — Dataset-Gated YOLO11 Baselines Before RT-DETR

This notebook intentionally trains **YOLO first** on the accepted six-class V6.2E split.

The purpose is not to crown YOLO as the final model. It is to establish that:

1. the frozen dataset reconstructs exactly;
2. all six classes are present with the expected support;
3. labels are valid;
4. a mature detector can learn the dataset without pathological behavior;
5. only after that do we move to RT-DETR B0/B1/B2/B3.

Accepted dataset:

```text
V6.2E tag: v6p2e_23562a45b343
Train / Val / Test: 17,483 / 2,272 / 2,012
```

Expected final class distribution from the accepted audit:

| class | train boxes | val boxes | test boxes |
|---|---:|---:|---:|
| insulator | 6,226 | 778 | 1,267 |
| insulator_broken | 6,236 | 791 | 809 |
| insulator_flashover | 1,249 | 163 | 124 |
| bird_nest | 6,085 | 821 | 178 |
| foreign_object | 667 | 98 | 161 |
| broken_strand | 2,002 | 252 | 134 |

The TEST split remains untouched during development.

## Training order

1. **YOLO11n @640** — quick sanity/edge baseline.
2. **YOLO11m @1024** — main practical baseline.
3. RT-DETR-R50 B0 only after these results and dataset QA look sane.

No RT-DETR is trained in this notebook.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile, time, threading
from collections import Counter, defaultdict

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p2e_yolo_baseline")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
RECON_LABELS = TMP/"reconciled_labels"
RUNTIME = TMP/"v6p2e_runtime"
RUNS = W/"v6p2e_yolo_runs"

for p in [TMP,BASE_EXTRACT,OVERLAY_EXTRACT,RECON_LABELS,RUNTIME,RUNS]:
    p.mkdir(parents=True,exist_ok=True)

FINAL_TAG = "v6p2e_23562a45b343"
PARENT_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
FINAL_ROOT = f"{DATA_ROOT}/v6p2e/{FINAL_TAG}"
PARENT_ROOT = f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"

FREEZE_STASH = f"{FINAL_ROOT}/v6p2e.freeze.json"
MANIFEST_STASH = f"{FINAL_ROOT}/v6p2e_split_manifest.csv"
RECON_STASH = f"{FINAL_ROOT}/v6p2e_reconciled_labels.zip"

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_SPLITS = {"train":17483,"val":2272,"test":2012}

EXPECTED_BOXES = {
    "train":[6226,6236,1249,6085,667,2002],
    "val":[778,791,163,821,98,252],
    "test":[1267,809,124,178,161,134],
}
EXPECTED_IMAGES = {
    "train":[2493,5125,1081,5423,549,1901],
    "val":[322,656,140,717,71,248],
    "test":[712,661,109,167,134,121],
}

TRAIN_YOLO11N = True
TRAIN_YOLO11M = True
RUN_FINAL_TEST = False

SEED = 42
WORKERS = 4
UPLOAD_EVERY_MIN = 15

CKPT_ROOT = (
    "stash://bunpmc/projects/yolov11sdi/checkpoints/"
    f"v6p2e/{FINAL_TAG}/yolo_baselines"
)

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}


## 1. Install runtime and verify 2×T4

In [ ]:
subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "ultralytics","pyyaml","pandas"
])

import pandas as pd
import yaml
import numpy as np
import torch

print("Torch:",torch.__version__)
print("GPU count:",torch.cuda.device_count())
print("GPUs:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

assert torch.cuda.device_count()>=2, "Enable Kaggle T4 x2."

def images_under(root):
    return sorted(
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

def sha256_file(path,chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()


## 2. Camber authentication

In [ ]:
from kaggle_secrets import UserSecretsClient

key=UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")

os.environ["CAMBER_API_KEY"]=key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"]=(
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src,dst,required=True):
    dst=Path(dst)
    dst.parent.mkdir(parents=True,exist_ok=True)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    ok=r.returncode==0 and dst.exists()
    if required and not ok:
        raise RuntimeError(
            f"Camber copy failed: {src}\n"
            f"{r.stdout[-1000:]}\n{r.stderr[-1600:]}"
        )
    return ok

def stash_put(src,dst,required=True):
    src=Path(src)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    ok=r.returncode==0
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    if ok:
        print("Uploaded:",dst)
    return ok


## 3. Verify the accepted V6.2E freeze

This is a hard gate. Training will not start if the requested tag is not the exact accepted freeze.


In [ ]:
freeze_local=TMP/"v6p2e.freeze.json"
manifest_local=TMP/"v6p2e_split_manifest.csv"
recon_zip=TMP/"v6p2e_reconciled_labels.zip"

stash_cp(FREEZE_STASH,freeze_local)
stash_cp(MANIFEST_STASH,manifest_local)
stash_cp(RECON_STASH,recon_zip)

freeze=json.loads(freeze_local.read_text())
manifest=pd.read_csv(manifest_local)

assert freeze["final_tag"]==FINAL_TAG
assert freeze["parent_dataset_tag"]==PARENT_TAG
assert freeze["freeze_ready"] is True
assert freeze["split_counts"]==EXPECTED_SPLITS

integrity=freeze["integrity"]
assert integrity["exact_cross_split_overlap"]==0
assert integrity["strong_neardup_cross_split_edges"]==0
assert integrity["medium_candidate_cross_split_edges"]==0
assert integrity["test_related_quarantine_remaining_in_trainval"]==0
assert integrity["test_hash_set_unchanged"] is True

print("✅ Freeze verified:",FINAL_TAG)
print(json.dumps(freeze["integrity"],indent=2))


## 4. Restore immutable base + FOTL overlay

In [ ]:
base_tar=TMP/f"base_{BASE_TAG}.tar"
overlay_tar=TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

if not any(BASE_EXTRACT.rglob("data.yaml")):
    stash_cp(BASE_STASH,base_tar)
    subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
    base_tar.unlink(missing_ok=True)

if not any(OVERLAY_EXTRACT.rglob("data.yaml")):
    stash_cp(OVERLAY_STASH,overlay_tar)
    subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])
    overlay_tar.unlink(missing_ok=True)

if RECON_LABELS.exists():
    shutil.rmtree(RECON_LABELS)
RECON_LABELS.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(recon_zip) as z:
    z.extractall(RECON_LABELS)
recon_zip.unlink(missing_ok=True)

def norm_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names)

def find_root(root):
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names",[]))==NAMES:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"No matching six-class dataset root under {root}")

BASE=find_root(BASE_EXTRACT)
OVERLAY=find_root(OVERLAY_EXTRACT)

print("BASE:",BASE)
print("OVERLAY:",OVERLAY)
print("Reconciled label files:",len(list(RECON_LABELS.rglob("*.txt"))))


## 5. Reconstruct the exact final split from the V6.2E manifest

The old absolute paths in the manifest came from the audit Kaggle session.  
This cell maps each manifest row back to the immutable base/overlay source using its old split and path suffix, then verifies image SHA256.


In [ ]:
def locate_original(row):
    old_path=str(row["image_path_source"]).replace("\\","/")
    old_split=str(row["previous_v6p1_split"])
    marker=f"/images/{old_split}/"

    if marker not in old_path:
        raise RuntimeError(f"Cannot parse manifest source path: {old_path}")

    suffix=old_path.split(marker,1)[1]

    if suffix.startswith("base/"):
        rel=suffix[len("base/"):]
        root=BASE
    elif suffix.startswith("fotl/"):
        rel=suffix[len("fotl/"):]
        root=OVERLAY
    else:
        raise RuntimeError(f"Unknown manifest source suffix: {suffix}")

    ip=root/"images"/old_split/rel
    lp=root/"labels"/old_split/Path(rel).with_suffix(".txt")

    if not ip.exists():
        raise FileNotFoundError(ip)

    return ip,lp

def locate_final_label(row,original_lp):
    # Reconciled exact-duplicate labels are archived by SHA.
    rp=RECON_LABELS/f"{row['sha256']}.txt"
    resolution=str(row.get("resolution",""))
    if resolution.startswith("merged"):
        if not rp.exists():
            raise FileNotFoundError(
                f"Missing reconciled label for {row['sha256']}"
            )
        return rp
    return original_lp

records=[]

for idx,row in manifest.iterrows():
    ip,original_lp=locate_original(row)

    got=sha256_file(ip)
    expected=str(row["sha256"])
    if got!=expected:
        raise RuntimeError(
            f"SHA mismatch row {idx}: {got} != {expected}"
        )

    lp=locate_final_label(row,original_lp)

    records.append({
        "split":str(row["split"]),
        "source":str(row["source"]),
        "sha256":got,
        "image_path":str(ip),
        "label_path":str(lp),
        "resolution":str(row.get("resolution","")),
    })

rec_df=pd.DataFrame(records)

actual=rec_df.groupby("split").size().to_dict()
print("Reconstructed counts:",actual)
assert actual==EXPECTED_SPLITS

print("✅ Exact manifest reconstructed.")


## 6. Label-integrity + exact distribution gate

This checks the reconstructed artifact itself, not merely metadata:

- every image has a label file;
- class IDs are valid;
- YOLO coordinates are normalized and positive;
- all six classes exist in train/val/test;
- class image and box counts exactly match the accepted V6.2E audit.


In [ ]:
def parse_label(path):
    boxes=[]
    path=Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    for line_no,line in enumerate(path.read_text(errors="ignore").splitlines(),1):
        if not line.strip():
            continue
        p=line.split()
        if len(p)<5:
            raise RuntimeError(f"Malformed label {path}:{line_no}")
        cid=int(float(p[0]))
        x,y,w,h=map(float,p[1:5])

        if not (0<=cid<len(NAMES)):
            raise RuntimeError(f"Bad class id {cid}: {path}:{line_no}")
        if not (0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1):
            raise RuntimeError(
                f"Bad normalized bbox {x,y,w,h}: {path}:{line_no}"
            )
        boxes.append((cid,x,y,w,h))
    return boxes

box_counts={sp:[0]*len(NAMES) for sp in EXPECTED_SPLITS}
img_counts={sp:[0]*len(NAMES) for sp in EXPECTED_SPLITS}
empty_labels=0

for i,r in enumerate(records,1):
    boxes=parse_label(r["label_path"])
    if not boxes:
        empty_labels+=1

    seen=set()
    for b in boxes:
        box_counts[r["split"]][b[0]]+=1
        seen.add(b[0])
    for cid in seen:
        img_counts[r["split"]][cid]+=1

    if i%5000==0:
        print("validated labels",i,"/",len(records))

rows=[]
for cid,name in enumerate(NAMES):
    rows.append({
        "class":name,
        "train_images":img_counts["train"][cid],
        "train_boxes":box_counts["train"][cid],
        "val_images":img_counts["val"][cid],
        "val_boxes":box_counts["val"][cid],
        "test_images":img_counts["test"][cid],
        "test_boxes":box_counts["test"][cid],
    })

dist=pd.DataFrame(rows)
display(dist)

for sp in ["train","val","test"]:
    assert box_counts[sp]==EXPECTED_BOXES[sp], (
        sp,box_counts[sp],EXPECTED_BOXES[sp]
    )
    assert img_counts[sp]==EXPECTED_IMAGES[sp], (
        sp,img_counts[sp],EXPECTED_IMAGES[sp]
    )
    assert all(n>0 for n in box_counts[sp])

print("Empty/background labels:",empty_labels)
print("✅ Label coordinates valid.")
print("✅ Six-class distribution exactly matches accepted V6.2E.")


## 7. Rebuild YOLO runtime

TEST is included in `data.yaml` but is not evaluated by the training cells.


In [ ]:
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)

for sp in ["train","val","test"]:
    (RUNTIME/"images"/sp).mkdir(parents=True,exist_ok=True)
    (RUNTIME/"labels"/sp).mkdir(parents=True,exist_ok=True)

for i,r in enumerate(records):
    sp=r["split"]
    src=Path(r["image_path"])
    lp=Path(r["label_path"])
    ext=src.suffix.lower()

    name=f"{r['sha256'][:16]}_{i:06d}{ext}"
    di=RUNTIME/"images"/sp/name
    dl=RUNTIME/"labels"/sp/Path(name).with_suffix(".txt")

    try:
        os.link(src,di)
    except Exception:
        os.symlink(src,di)

    try:
        os.link(lp,dl)
    except Exception:
        os.symlink(lp,dl)

(RUNTIME/"data.yaml").write_text(yaml.safe_dump({
    "path":str(RUNTIME),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":NAMES,
},sort_keys=False))

print((RUNTIME/"data.yaml").read_text())
print({
    sp:len(images_under(RUNTIME/"images"/sp))
    for sp in ["train","val","test"]
})


## 8. Training helpers with Camber resume

Checkpoints are stored under the **V6.2E final tag**, so later RT-DETR experiments can cite the same dataset lineage.


In [ ]:
from ultralytics import YOLO

class Uploader(threading.Thread):
    def __init__(self,local,remote,every_min=15):
        super().__init__(daemon=True)
        self.local=Path(local)
        self.remote=remote
        self.every=every_min*60
        self.stop_ev=threading.Event()
        self.last_mtime=None

    def push(self):
        if not self.local.exists():
            return
        m=self.local.stat().st_mtime
        if m==self.last_mtime:
            return

        tmp=self.local.with_suffix(".upload.pt")
        shutil.copy2(self.local,tmp)
        try:
            stash_put(tmp,self.remote)
            self.last_mtime=m
        finally:
            tmp.unlink(missing_ok=True)

    def run(self):
        while not self.stop_ev.wait(self.every):
            self.push()

    def stop(self):
        self.stop_ev.set()
        self.push()

def train_resumable(
    name,
    base_model,
    imgsz,
    epochs,
    batch,
    lr0,
    patience,
):
    run=RUNS/name
    wdir=run/"weights"
    wdir.mkdir(parents=True,exist_ok=True)

    best_l=wdir/"best.pt"
    last_l=wdir/"last.pt"
    remote=f"{CKPT_ROOT}/{name}"
    done_l=TMP/f"{name}.done.json"

    if stash_cp(f"{remote}/done.json",done_l,required=False):
        stash_cp(f"{remote}/best.pt",best_l,required=True)
        print(f"[{name}] already complete -> reuse best.pt")
        return best_l,json.loads(done_l.read_text())

    resume=last_l.exists() or stash_cp(
        f"{remote}/last.pt",
        last_l,
        required=False
    )

    up=Uploader(
        last_l,
        f"{remote}/last.pt",
        UPLOAD_EVERY_MIN
    )
    up.start()

    t0=time.time()

    try:
        if resume:
            print(f"[{name}] RESUME")
            YOLO(str(last_l)).train(resume=True)
        else:
            print(f"[{name}] FRESH TRAIN")
            YOLO(base_model).train(
                data=str(RUNTIME/"data.yaml"),
                project=str(RUNS),
                name=name,
                exist_ok=True,
                device=[0,1],
                seed=SEED,
                workers=WORKERS,
                deterministic=False,

                imgsz=imgsz,
                epochs=epochs,
                batch=batch,

                optimizer="AdamW",
                lr0=lr0,
                lrf=0.10,
                cos_lr=True,
                warmup_epochs=1.0,
                warmup_bias_lr=0.0,
                weight_decay=5e-4,
                patience=patience,

                amp=True,
                cache=False,
                plots=False,

                mosaic=0.25,
                close_mosaic=5,
                mixup=0.0,
                copy_paste=0.0,
                scale=0.15,
                translate=0.05,
                hsv_h=0.01,
                hsv_s=0.35,
                hsv_v=0.25,
                fliplr=0.5,
            )
    finally:
        up.stop()

    if not best_l.exists() and last_l.exists():
        shutil.copy2(last_l,best_l)
    assert best_l.exists()

    info={
        "name":name,
        "base_model":base_model,
        "dataset_tag":FINAL_TAG,
        "classes":NAMES,
        "split_counts":EXPECTED_SPLITS,
        "imgsz":imgsz,
        "epochs":epochs,
        "batch":batch,
        "lr0":lr0,
        "seconds":round(time.time()-t0),
    }

    csvp=run/"results.csv"
    if csvp.exists():
        rows=list(csv.DictReader(open(csvp)))
        if rows:
            info["last_epoch"]={
                k.strip():v.strip()
                for k,v in rows[-1].items()
            }

    stash_put(best_l,f"{remote}/best.pt")
    if last_l.exists():
        stash_put(last_l,f"{remote}/last.pt")
    if csvp.exists():
        stash_put(csvp,f"{remote}/results.csv")

    done_l.write_text(json.dumps(info,indent=2))
    stash_put(done_l,f"{remote}/done.json")

    return best_l,info


## 9. YOLO11n @640 sanity baseline

This is intentionally first. It is cheap enough to expose broken labels/splits before spending more GPU time.


In [ ]:
YOLO11N_BEST=None

if TRAIN_YOLO11N:
    YOLO11N_BEST,info_n=train_resumable(
        name="yolo11n_v6p2e_640",
        base_model="yolo11n.pt",
        imgsz=640,
        epochs=50,
        batch=64,
        lr0=1e-3,
        patience=10,
    )
    print(json.dumps(info_n,indent=2)[:6000])


## 10. YOLO11m @1024 main practical baseline


In [ ]:
YOLO11M_BEST=None

if TRAIN_YOLO11M:
    YOLO11M_BEST,info_m=train_resumable(
        name="yolo11m_v6p2e_1024",
        base_model="yolo11m.pt",
        imgsz=1024,
        epochs=40,
        batch=16,
        lr0=3e-4,
        patience=8,
    )
    print(json.dumps(info_m,indent=2)[:6000])


## 11. VAL-only evaluation with explicit six-class coverage

The evaluator hard-stops if any class is absent from the evaluated validation set.


In [ ]:
def evaluate_val(best,imgsz,batch):
    if best is None:
        return None

    m=YOLO(str(best)).val(
        data=str(RUNTIME/"data.yaml"),
        split="val",
        imgsz=imgsz,
        batch=batch,
        device=0,
        plots=False,
        verbose=False,
    )

    eval_ids=[int(x) for x in np.asarray(m.box.ap_class_index).tolist()]
    missing=sorted(set(range(len(NAMES)))-set(eval_ids))
    if missing:
        raise RuntimeError(
            "Validation missing evaluated classes: "
            + str([(i,NAMES[i]) for i in missing])
        )

    ap50=np.asarray(m.box.ap50)
    ap=np.asarray(m.box.ap).mean(axis=1)

    per_class={}
    for pos,cid in enumerate(eval_ids):
        per_class[NAMES[cid]]={
            "mAP50":float(ap50[pos]),
            "mAP50_95":float(ap[pos]),
        }

    return {
        "precision":float(m.box.mp),
        "recall":float(m.box.mr),
        "mAP50":float(m.box.map50),
        "mAP50_95":float(m.box.map),
        "per_class":per_class,
    }

results={
    "dataset_tag":FINAL_TAG,
    "split":"val",
    "test_touched":False,
    "expected_split_counts":EXPECTED_SPLITS,
}

if YOLO11N_BEST:
    results["YOLO11n_640"]=evaluate_val(
        YOLO11N_BEST,640,32
    )

if YOLO11M_BEST:
    results["YOLO11m_1024"]=evaluate_val(
        YOLO11M_BEST,1024,16
    )

print(json.dumps(results,indent=2))

result_path=W/f"{FINAL_TAG}_YOLO11_VAL_BASELINES.json"
result_path.write_text(json.dumps(results,indent=2))
stash_put(result_path,f"{CKPT_ROOT}/{result_path.name}")


## 12. Optional final TEST evaluation

Leave this `False` until the baseline recipe is frozen.

Do not use TEST to choose architecture, image size, augmentation, epochs, or learning rate.


In [ ]:
if RUN_FINAL_TEST:
    def eval_test(best,imgsz,batch):
        m=YOLO(str(best)).val(
            data=str(RUNTIME/"data.yaml"),
            split="test",
            imgsz=imgsz,
            batch=batch,
            device=0,
            plots=False,
            verbose=False,
        )
        eval_ids=[int(x) for x in np.asarray(m.box.ap_class_index).tolist()]
        missing=sorted(set(range(len(NAMES)))-set(eval_ids))
        if missing:
            raise RuntimeError(
                "TEST missing classes: "
                + str([(i,NAMES[i]) for i in missing])
            )

        ap50=np.asarray(m.box.ap50)
        ap=np.asarray(m.box.ap).mean(axis=1)
        per_class={
            NAMES[cid]:{
                "mAP50":float(ap50[pos]),
                "mAP50_95":float(ap[pos]),
            }
            for pos,cid in enumerate(eval_ids)
        }
        return {
            "precision":float(m.box.mp),
            "recall":float(m.box.mr),
            "mAP50":float(m.box.map50),
            "mAP50_95":float(m.box.map),
            "per_class":per_class,
        }

    test_results={
        "dataset_tag":FINAL_TAG,
        "FINAL_TEST_DO_NOT_TUNE":True,
    }

    if YOLO11N_BEST:
        test_results["YOLO11n_640"]=eval_test(
            YOLO11N_BEST,640,32
        )
    if YOLO11M_BEST:
        test_results["YOLO11m_1024"]=eval_test(
            YOLO11M_BEST,1024,16
        )

    print(json.dumps(test_results,indent=2))
    p=W/f"{FINAL_TAG}_FINAL_YOLO11_TEST.json"
    p.write_text(json.dumps(test_results,indent=2))
    stash_put(p,f"{CKPT_ROOT}/{p.name}")
else:
    print("✅ TEST untouched.")


# Decision point before RT-DETR

Proceed to RT-DETR-R50 B0 only after checking:

- dataset reconstruction/distribution gates passed;
- YOLO11n learns all six classes without pathological collapse;
- YOLO11m gives a sensible improvement or at least a coherent per-class pattern;
- `foreign_object` and `insulator_flashover` are interpreted with their smaller support in mind;
- TEST has remained untouched.

Then RT-DETR B0 should use this exact `v6p2e_23562a45b343` split.

If RT-DETR underperforms after the same frozen dataset is shown to be learnable by YOLO, blaming the architecture/training recipe becomes considerably less embarrassing.
